In [3]:
!pip install beautifulsoup4
!pip install openpyxl
!pip install yfinance

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ---------------------------------------- 0.0/3.0 MB ? eta -:--:--
     --- ------------------------------------ 0.3/3.0 MB ? eta -:--:--
     ---------- ----------------------------- 0.8/3.0 MB 1.8 MB/s eta 0:00:02
     -------------------- ------------------- 1.6/3.0 MB 2.5 MB/s eta 0:00:01
     --------------------------- ------------ 2.1/3.0 MB 2.5 MB/s eta 0:00:01
     ---------------------------------------- 3.0/3.0 MB 3.1 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------------------- -------------------- 0

  DEPRECATION: Building 'multitasking' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'multitasking'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [4]:
import io
import json
import os
import re
from datetime import timedelta

import gspread
import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from gspread.cell import Cell

load_dotenv()

SPREADSHEET_ID = os.getenv("SPREADSHEET_ID", "1WlamXyzIj6GZAkU_lc8C0mTvMzwoHZk-R_HodUC3Sws")
WORKSHEET_NAME = os.getenv("WORKSHEET_NAME", "list")
JPX_PAGE = "https://www.jpx.co.jp/listing/event-schedules/financial-announcement/"
SERVICE_ACCOUNT_FILE = os.getenv("GOOGLE_CREDENTIALS_FILE") or os.getenv("GOOGLE_APPLICATION_CREDENTIALS", "abiding-ascent-476815-q6-56a05b29f113.json")
CALENDAR_ID = os.getenv("GOOGLE_CALENDAR_ID", "primary")
SHEETS_SCOPES = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
CALENDAR_SCOPES = ["https://www.googleapis.com/auth/calendar"]


def pick_column(columns, keywords):
    for col in columns:
        label = str(col)
        if any(k in label for k in keywords):
            return col
    raise ValueError(f"Column not found for {keywords}")


def pick_header_index(headers, candidates):
    for name in candidates:
        if name in headers:
            return headers.index(name)
    return None


def normalize_code(raw):
    if not raw:
        return None
    m = re.search(r"\d{4}", str(raw))
    return m.group(0).zfill(4) if m else None


def fetch_all_jpx_df():
    resp = requests.get(JPX_PAGE, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    base = "https://www.jpx.co.jp"
    xlsx_links = [
        base + a["href"]
        for a in soup.find_all("a", href=True)
        if a["href"].endswith(".xlsx")
    ]
    if not xlsx_links:
        raise RuntimeError("JPX????xlsx??????????????")

    dfs = []
    for url in xlsx_links:
        try:
            file_resp = requests.get(url, timeout=30)
            file_resp.raise_for_status()
            df = pd.read_excel(io.BytesIO(file_resp.content), engine="openpyxl", skiprows=4)
            df.columns = [str(c).split("\n")[0].strip() for c in df.columns]
            dfs.append(df)
        except Exception as e:
            print(f"??????: {url} ({e})")

    if not dfs:
        raise RuntimeError("xlsx???????????")

    return pd.concat(dfs, ignore_index=True)


def fetch_close_price(code: str):
    if not code:
        return None
    ticker = f"{code}.T"
    try:
        data = yf.download(ticker, period="5d", interval="1d", auto_adjust=False, progress=False)
        if data.empty:
            return None

        close = data["Close"]
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
        close = close.dropna()
        if close.empty:
            return None

        return float(close.iloc[-1])
    except Exception:
        return None


def build_earnings_calendar():
    df = fetch_all_jpx_df()

    code_col = pick_column(df.columns, ["???"])
    date_col = pick_column(df.columns, ["???????", "?????"])
    kind_col = pick_column(df.columns, ["??", "??"])

    df[code_col] = df[code_col].map(normalize_code)
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df[kind_col] = df[kind_col].astype(str).str.strip()

    calendar = (
        df.dropna(subset=[code_col])
        .drop_duplicates(subset=[code_col], keep="first")
        .set_index(code_col)[[date_col, kind_col]]
    )
    return calendar


def build_calendar_service():
    creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=CALENDAR_SCOPES)
    return build("calendar", "v3", credentials=creds, cache_discovery=False)


def extract_http_reason(error: HttpError):
    try:
        payload = json.loads(error.content.decode("utf-8"))
        return payload["error"]["errors"][0].get("reason", "")
    except Exception:
        return ""


def event_exists(calendar_service, calendar_id, summary, date_str):
    next_date = (pd.to_datetime(date_str) + timedelta(days=1)).strftime("%Y-%m-%d")
    time_min = f"{date_str}T00:00:00Z"
    time_max = f"{next_date}T00:00:00Z"

    result = calendar_service.events().list(
        calendarId=calendar_id,
        timeMin=time_min,
        timeMax=time_max,
        singleEvents=True,
        q=summary,
        maxResults=20,
    ).execute()

    for event in result.get("items", []):
        start_date = event.get("start", {}).get("date")
        event_summary = (event.get("summary") or "").strip()
        if start_date == date_str and event_summary == summary:
            return True

    return False


def create_all_day_event(calendar_service, calendar_id, summary, date_str, description=None):
    next_date = (pd.to_datetime(date_str) + timedelta(days=1)).strftime("%Y-%m-%d")
    body = {
        "summary": summary,
        "start": {"date": date_str},
        "end": {"date": next_date},
    }
    if description:
        body["description"] = description
    calendar_service.events().insert(calendarId=calendar_id, body=body).execute()


def main():
    sheets_creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SHEETS_SCOPES)
    gc = gspread.authorize(sheets_creds)
    ws = gc.open_by_key(SPREADSHEET_ID).worksheet(WORKSHEET_NAME)

    values = ws.get_all_values()
    if not values:
        return

    headers = values[0]
    try:
        code_idx = headers.index("?????")
        date_idx = headers.index("?????")
        kind_idx = headers.index("????")
        price_idx = headers.index("????")
    except ValueError as e:
        raise RuntimeError("???????????????????????????????????????????") from e

    name_idx = pick_header_index(headers, ["???", "???", "???"])
    if name_idx is None:
        print("??????(???/???/???)??????????Calendar????????????")

    earnings_calendar = build_earnings_calendar()
    calendar_service = build_calendar_service()
    calendar_disabled = False
    created_events = 0

    updates = []
    for row_num, row in enumerate(values[1:], start=2):
        code = normalize_code(row[code_idx] if code_idx < len(row) else "")
        if not code:
            continue

        if code in earnings_calendar.index:
            date_val, kind_val = earnings_calendar.loc[code]
            date_str = None

            if not pd.isna(date_val):
                date_str = date_val.strftime("%Y-%m-%d")
                updates.append(Cell(row=row_num, col=date_idx + 1, value=date_str))

            if isinstance(kind_val, str) and kind_val:
                updates.append(Cell(row=row_num, col=kind_idx + 1, value=kind_val))

            company_name = ""
            if name_idx is not None and name_idx < len(row):
                company_name = str(row[name_idx]).strip()

            if (not calendar_disabled) and company_name and date_str and isinstance(kind_val, str) and kind_val:
                summary = f"{company_name}{kind_val}"
                description = f"?????: {code}"
                try:
                    if not event_exists(calendar_service, CALENDAR_ID, summary, date_str):
                        create_all_day_event(
                            calendar_service=calendar_service,
                            calendar_id=CALENDAR_ID,
                            summary=summary,
                            date_str=date_str,
                            description=description,
                        )
                        created_events += 1
                except HttpError as e:
                    reason = extract_http_reason(e)
                    if reason == "accessNotConfigured":
                        calendar_disabled = True
                        print(
                            "Calendar API???????"
                            "Google Cloud? Calendar API ????????????????"
                        )
                    elif reason in {"notFound", "forbidden"}:
                        calendar_disabled = True
                        print(
                            f"Calendar ID '{CALENDAR_ID}' ???????????"
                            "??????????GOOGLE_CALENDAR_ID??????????"
                        )
                    else:
                        print(f"Calendar????: code={code}, date={date_str}, error={e}")

        close_price = fetch_close_price(code)
        if close_price is not None:
            updates.append(Cell(row=row_num, col=price_idx + 1, value=close_price))

    if updates:
        ws.update_cells(updates, value_input_option="USER_ENTERED")

    print(f"Google Calendar? {created_events} ??????? (calendar_id={CALENDAR_ID})")


if __name__ == "__main__":
    main()



C:\Users\ryuho.nakano\AppData\Local\Temp\ipykernel_31188\3696499304.py:71: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period="5d", interval="1d", progress=False)
C:\Users\ryuho.nakano\AppData\Local\Temp\ipykernel_31188\3696499304.py:75: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return float(data["Close"].iloc[-1])
C:\Users\ryuho.nakano\AppData\Local\Temp\ipykernel_31188\3696499304.py:71: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period="5d", interval="1d", progress=False)
C:\Users\ryuho.nakano\AppData\Local\Temp\ipykernel_31188\3696499304.py:75: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  return float(data["Close"].iloc[-1])
C:\Users\ryuho.nakano\AppData\Loca